# Autonomous Grid Balancing Demo

This notebook demonstrates the `orchestrator_agent` running a multi-agent loop to balance a simulated grid.

In [ ]:
import sys
import os

# 1. Navigate to 'automated-grid-balancing' root to simplify imports
notebook_dir = os.getcwd()
if notebook_dir.endswith('notebooks'):
    # We are in notebooks/, go up
    grid_root = os.path.abspath(os.path.join(notebook_dir, '..'))
    os.chdir(grid_root)
    print(f"Changed working directory to: {grid_root}")
else:
    grid_root = notebook_dir
    print(f"Working directory is: {grid_root}")

# 2. Add AgentField root (for agentfield.py)
project_root = os.path.abspath(os.path.join(grid_root, '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project Root: {project_root}")

from agentfield import app

# 3. Import agents directly (now that CWD is grid_root)
from agents.orchestrator_agent import OrchestratorAgent
from agents.telemetry_agent import TelemetryAgent
from agents.forecast_agent import ForecastAgent
from agents.policy_agent import PolicyAgent
from agents.planner_agent import PlannerAgent
from agents.verifier_agent import VerifierAgent

from common.schemas import DatasetConfig, RunRequest
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# 1. Setup Run Request
# Data is in AgentField/data
data_dir = os.path.abspath(os.path.join(project_root, 'data'))

dataset = DatasetConfig(
    pjm_dir=data_dir,
    pjm_pattern="eia_hourly.csv", 
    region="PJM"
)

req = RunRequest(
    dataset=dataset,
    horizon_steps=12,
    n_steps=24 
)

In [ ]:
# 2. Run Orchestrator
orchestrator = OrchestratorAgent()
result = orchestrator.plan_run(req)

print("Run Complete!")
print(f"Summary: {result.summary}")
print(f"Artifacts: {result.artifacts}")

In [ ]:
# 3. Analyze Results
import json

with open(result.artifacts['audits'], 'r') as f:
    audits = json.load(f)

df_audit = pd.DataFrame([
    {
        'time': a['timestamp'],
        'cost': a['cost'],
        'battery': a['action']['battery_mw'],
        'peaker': a['action']['peaker_mw'],
        'curtail': a['action']['curtail_mw']
    } for a in audits
])
df_audit['time'] = pd.to_datetime(df_audit['time'])
df_audit.set_index('time', inplace=True)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
df_audit[['battery', 'peaker']].plot(kind='bar', stacked=True, ax=ax)
plt.title("Grid Actions Over Time")
plt.ylabel("MW")
plt.show()